# Fine-tune signalbench on Colab or a local NVIDIA laptop

This notebook runs the repository's **same CLI and training code**. It is an execution interface,
not a second research implementation. No training results are supplied in advance.

**Recommended:** Colab GPU runtime (T4/L4 or better), profile `hosted_3b`. A free GPU is not guaranteed.
Select **Runtime → Change runtime type → GPU** before proceeding. For a rented Hugging Face GPU,
use a notebook server on persistent storage or follow `docs/TRAINING.md` for the Jobs distinction.

**Local:** Linux or WSL2 with NVIDIA CUDA drivers, Python 3.11–3.13 and a Jupyter kernel in your venv.
Choose `rtx4060` for the primary 3B experiment. Assuming you mean GTX 1650, choose `gtx1650` for a
**separate exploratory 0.5B experiment**, not a claimed 3B run. Actual GPU memory is checked.
An overnight run cannot fix insufficient VRAM. The small profile uses FP32 compute with NF4 weights
for older hardware; it may be slow. Run the smoke test first; fit and duration are not guaranteed.

The notebook never calls a paid inference API, launches a paid GPU job, or publishes weights automatically.
The API inference budget does not cover GPU rentals. Base and dataset research licenses still apply.


In [ ]:
# 1. Locate/clone the source. For an exact replay set SOURCE_REF to a recorded full commit SHA.
import os
import subprocess
import sys
from pathlib import Path

SOURCE_REF = None
REPOSITORY = "https://github.com/nishsm/signalbench.git"
try:
    __import__("google.colab")

    IS_COLAB = True
except ImportError:
    IS_COLAB = False

REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/signalbench").is_dir()), None
)
if REPO is None:
    REPO = Path("/content/signalbench") if IS_COLAB else Path.cwd() / "signalbench"
    subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    if SOURCE_REF:
        subprocess.run(["git", "-C", str(REPO), "checkout", SOURCE_REF], check=True)
os.chdir(REPO)
SOURCE_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if SOURCE_REF and SOURCE_SHA != SOURCE_REF:
    raise RuntimeError(
        "Existing checkout differs from SOURCE_REF; select the intended checkout manually"
    )
print("Repository:", REPO)
print("Record this source commit for replay:", SOURCE_SHA)

## Install once
Use a fresh kernel. If you already imported torch/transformers before this cell, restart the kernel
and rerun from the first cell after installation. Installing into Colab can take several minutes.
Pinned PyTorch supplies its CUDA runtime, but the machine still needs a compatible NVIDIA driver.
For a local run, use a dedicated environment, not the system Python.


In [ ]:
# 2. Pinned dependencies; optional local Trackio monitoring is included.
INSTALL_DEPENDENCIES = True
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use a Python 3.11, 3.12 or 3.13 kernel")
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[ml,train,tracking]"], check=True
    )
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

## Select hardware and persistent output storage
For Colab, Google Drive keeps checkpoints, the adapter and results across runtime disconnects.
A disconnect can interrupt training; rerunning the same profile/root resumes the latest completed
checkpoint. Colab does not guarantee an uninterrupted overnight session. Base-weight caches stay
on the runtime disk and may need downloading again after a reset. Drive checkpoint writes are slower.

Locally, leave the run folder on a disk with free space for optimizer checkpoints. Use a new run folder
when changing hyperparameters. Keep the laptop plugged in, prevent sleep and maintain cooling.


In [ ]:
import json

import yaml

from signalbench.core import artifact, config
from signalbench.models.training import hardware, prepare_run

# 3. Change PROFILE only here. Do not switch model size silently after an OOM.
PROFILE = "hosted_3b" if IS_COLAB else "rtx4060"  # or "gtx1650"
USE_GOOGLE_DRIVE = IS_COLAB
ENABLE_TRACKIO = True  # local metrics store; no remote Space is created
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/signalbench-runs")
else:
    RUN_ROOT = REPO / "runs"  # local persistent folder, gitignored


CONFIG_PATH = prepare_run(PROFILE, RUN_ROOT)
cfg = config(str(CONFIG_PATH))
cfg["training"]["tracking"] = "trackio" if ENABLE_TRACKIO else "none"
artifact(CONFIG_PATH, cfg, yaml.safe_dump(cfg, sort_keys=False))
os.environ["HF_HOME"] = str(Path(cfg["paths"]["cache"]) / "huggingface")
RUN_DIR = CONFIG_PATH.parent
print("Config:", CONFIG_PATH)
print("Model:", cfg["training"]["base"])
print("Persistent run folder:", RUN_DIR)

In [ ]:
# 4. Optional credentials. Public Qwen downloads do not require a token.
# Set True only if using authenticated Hub access. Input is hidden and kept in memory.
USE_HF_TOKEN = False
if USE_HF_TOKEN:
    from getpass import getpass

    os.environ["HF_TOKEN"] = getpass("Hugging Face token (hidden): ")

In [ ]:
# 5. Preflight before downloading weights. This is not a memory-fit guarantee.
import torch

info = hardware(cfg)
info.update(
    python=sys.version, torch=torch.__version__, cuda=torch.version.cuda, source_commit=SOURCE_SHA
)
artifact(
    Path(cfg["paths"]["results"]) / "training/environment.json", cfg, json.dumps(info, indent=2)
)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
artifact(Path(cfg["paths"]["results"]) / "training/environment.txt", cfg, freeze)


# Stream subprocess output; use the same commands locally in a terminal.
def stage(name, *extra):
    command = [
        sys.executable,
        "-u",
        "-m",
        "signalbench.cli",
        "--config",
        str(CONFIG_PATH),
        name,
        *extra,
    ]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, check=True, env=os.environ.copy())

## Data and fixed split check
This downloads the pinned public datasets and recomputes deduplication and splits with the repository
pipeline. The IDs must match the committed benchmark manifest. No test set is used to pick epochs.


In [ ]:
from signalbench.models.training import verify_saved_splits

# 6. Safe to rerun: immutable dataset revisions and local download cache are reused.
stage("data")

verify_saved_splits(cfg)
print("Split IDs match the committed research split.")

## Smoke test before the full run
This performs two optimizer updates, validation, adapter save and checkpoint save on a small
training/validation subset. Smoke outputs use separate folders and must not become research results.
If this fails, read the last error. For CUDA OOM, stop the process/kernel, free GPU memory and use a
larger GPU for 3B, or explicitly choose the separate small-model profile. Do not delete a full run's
checkpoints to force an incompatible resume. NaN/Inf losses must be investigated before proceeding.


In [ ]:
from signalbench.models.training import smoke_config

# 7. A failed subprocess raises and stops Run all before the full run.
stage("finetune", "--smoke")

smoke_results = Path(smoke_config(cfg)["paths"]["results"])
print((smoke_results / "training/runtime.json").read_text())

In [ ]:
# 8. Full training, validation-loss early stopping and automatic compatible-checkpoint resume.
# No test/holdout metrics are consulted during training.
stage("finetune")
print((Path(cfg["paths"]["results"]) / "training/runtime.json").read_text())

In [ ]:
# 9. Inspect the actual learning curves.
from IPython.display import Image, display

results = Path(cfg["paths"]["results"])
display(Image(filename=str(results / "training/curves.png")))

## Evaluate after choosing the training setup
This compares the untouched base and the adapter using identical label log-probability scoring on
the fixed validation, test, OOD and secondary partitions. It does not run paid API calls or the SEC
pipeline. Keep model-size/profile labels with results. Do not tune repeatedly against the test set.
These results alone do not establish a stock-return signal.


In [ ]:
import pandas as pd

# 10. This can take substantial time on an older GPU.
stage("bench")

table = pd.read_csv(results / "benchmark.csv")
display(table)

## Export and recover
The archive contains only adapter/tokenizer files, the run config, license notices and generated
results. It excludes raw data, secrets, base weights and optimizer states. The live checkpoint folder
remains in your persistent run folder for resumption. Download the ZIP before discarding a runtime.
Nothing is uploaded publicly here. Review the evaluation and licenses before publishing to the Hub.


In [ ]:
from signalbench.models.training import bundle_run

# 11. Local export; no public upload.

ARCHIVE = bundle_run(cfg, RUN_DIR / f"signalbench-{PROFILE}-adapter-results.zip")
print("Saved:", ARCHIVE)
if IS_COLAB:
    from google.colab import files

    files.download(str(ARCHIVE))

## Alternatives and sources
- [Hugging Face Jobs](https://huggingface.co/docs/hub/jobs-overview): managed GPU jobs, rather than
  a notebook runtime. Review eligibility, current GPU pricing, timeout and persistent Hub storage
  before submitting a paid job. This notebook does not submit one.
- [Tinker supported models](https://tinker-docs.thinkingmachines.ai/tinker/models/): managed LoRA
  training. The checked catalog does not list the primary Qwen2.5-3B; changing to a supported model
  requires a separately named experiment and an integration for exported adapters and scoring.
- [Colab limits](https://research.google.com/colaboratory/faq.html): GPUs and session lengths vary.
- [bitsandbytes requirements](https://huggingface.co/docs/bitsandbytes/v0.46.0/en/installation).
- [Qwen 3B license](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/blob/main/LICENSE) and
  [Qwen 0.5B license](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/LICENSE).

See `docs/TRAINING.md` for local overnight commands and checkpoint recovery.
